# 03 — Statistical Models: Baselines, SARIMAX and Optional Prophet

This notebook builds interpretable statistical forecasting models.

A serious forecasting PoC should always include baselines. More complex models only add value if they beat simple and robust alternatives under temporal validation.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

pd.set_option("display.max_columns", 80)
plt.rcParams["figure.figsize"] = (11, 4)


In [ ]:
from clinic_forecast.data import generate_synthetic_healthcare_data
from clinic_forecast.metrics import compute_metrics, metrics_by_group
from clinic_forecast.models.baseline import moving_average_forecast, seasonal_naive_forecast
from clinic_forecast.models.sarimax import sarimax_panel_forecast
from clinic_forecast.validation import rolling_origin_windows, split_window

data_path = PROJECT_ROOT / "data" / "raw" / "clinic_usage.csv"
if data_path.exists():
    usage = pd.read_csv(data_path, parse_dates=["date"])
else:
    usage, _, _ = generate_synthetic_healthcare_data()
    usage["date"] = pd.to_datetime(usage["date"])

windows = list(rolling_origin_windows(usage, horizon_days=28, n_windows=3, min_train_days=365))
train, test = split_window(usage, windows[-1])


## Seasonal naive baseline

The seasonal naive model repeats the last observed weekly pattern. It is simple, hard to beat, and a useful benchmark for daily operational data.


In [ ]:
seasonal_fcst = seasonal_naive_forecast(train=train, future=test, season_length=7)
scored_seasonal = test.merge(seasonal_fcst, on=["clinic_id", "date"], how="left")
compute_metrics(scored_seasonal["visits"], scored_seasonal["forecast"])


## Moving-average baseline

This model forecasts each clinic using its recent average demand. It can perform well when demand is stable but struggles with strong weekday effects.


In [ ]:
ma_fcst = moving_average_forecast(train=train, future=test, window=28)
scored_ma = test.merge(ma_fcst, on=["clinic_id", "date"], how="left")
compute_metrics(scored_ma["visits"], scored_ma["forecast"])


## SARIMAX with marketing variables

SARIMAX can model autoregressive structure, weekly seasonality and exogenous variables such as marketing spend and campaign activity.

For a portfolio PoC, this is a good model to show statistical modelling maturity because it is interpretable and operationally credible.


In [ ]:
# To keep the notebook responsive, start with a subset of clinics.
selected_clinics = ["CLINIC_001", "CLINIC_002", "CLINIC_003"]
train_small = train[train["clinic_id"].isin(selected_clinics)]
test_small = test[test["clinic_id"].isin(selected_clinics)]

sarimax_fcst = sarimax_panel_forecast(
    train=train_small,
    future=test_small,
    exog_cols=["marketing_spend", "campaign_active"],
)
scored_sarimax = test_small.merge(sarimax_fcst, on=["clinic_id", "date"], how="left")
compute_metrics(scored_sarimax["visits"], scored_sarimax["forecast"])


In [ ]:
from clinic_forecast.visualization import plot_actual_vs_forecast

one = scored_sarimax[scored_sarimax["clinic_id"] == selected_clinics[0]]
plot_actual_vs_forecast(one, title=f"SARIMAX forecast — {selected_clinics[0]}")


## Optional Prophet model

Prophet is optional because it can be heavier to install. Keep it isolated so that the core PoC remains runnable.


In [ ]:
try:
    from clinic_forecast.models.optional_prophet import prophet_forecast_one_clinic

    clinic_id = "CLINIC_001"
    train_one = train[train["clinic_id"] == clinic_id]
    prophet_fcst = prophet_forecast_one_clinic(train_one, periods=28)
    prophet_fcst.head()
except ImportError as exc:
    print(exc)
    print("Skipping Prophet example. Install optional dependencies to enable this cell.")
